In [15]:
import pandas as pd
import numpy as np

# 1. Load Data
df = pd.read_csv('data/semi-processed_2.csv')

# 2. Data Cleaning
# Standardize 'Type' and handle missing values in material columns
df['Type'] = df['Type'].replace('Individual', 'individual')
df['Secondary Material'] = df['Secondary Material'].fillna('None')
df['Composition'] = df['Composition'].fillna('None')

# Drop original raw numeric columns to avoid confusion
raw_cols = ['OP_Num', 'OTR_Num', 'PCO2_Num', 'WVP_Num', 'WVTR_Num']
df.drop(columns=[c for c in raw_cols if c in df.columns], inplace=True, errors='ignore')

# 3. Unit Conversion & Standardization
def standardize_units(df):
    # Oxygen Permeability (OP) Conversions
    # Mass (g) to Volume (cm3) using O2 density (~0.001429 g/cm3)
    o2_mass_cond = df['OP_Unit'] == 'gm/m2Pas'
    df.loc[o2_mass_cond, 'OP_Updated_Num'] *= (100 / 0.001429)
    df.loc[o2_mass_cond, 'OP_Unit'] = 'cm3cm/m2Pas'

    # Convert OP to a standard permeability format (cm3.um / m2.day.atm)
    # Conversion factors: 10000 (cm to um), 86400 (s to day), 101325 (Pa to atm)
    op_std_cond = df['OP_Unit'] == 'cm3cm/m2Pas'
    df.loc[op_std_cond, 'OP_Updated_Num'] *= (10000 * 86400 * 101325)
    df.loc[op_std_cond, 'OP_Unit'] = 'cm3um/m2dayatm'

    # Water Vapor Permeability (WVP) Conversions
    # Convert g/m2.Pa.s to g.um/m2.day.kPa
    wvp_std_cond = df['WVP_Unit'] == 'gm/m2Pas'
    df.loc[wvp_std_cond, 'WVP_Updated_Num'] *= (86400 * 1e6 * 1000)
    df.loc[wvp_std_cond, 'WVP_Unit'] = 'gum/m2daykPa'
    
    return df

df = standardize_units(df)

# 4. Transmission Rate Calculations
# Fill missing thickness with a standard benchmark (100 um or 0.0001 m)
df['thickness_Updated_Num'] = df['thickness_Updated_Num'].fillna(0.0001)

def calculate_rates(row):
    # Calculate OTR from standardized OP
    if pd.notnull(row['OP_Updated_Num']) and row['OP_Unit'] == 'cm3um/m2dayatm':
        row['OTR_Updated_Num'] = row['OP_Updated_Num'] / (row['thickness_Updated_Num'] * 1e6)
        row['OTR_Updated_Unit'] = 'cm3/m2day'
    
    # Calculate WVTR from standardized WVP
    if pd.notnull(row['WVP_Updated_Num']) and row['WVP_Unit'] == 'gum/m2daykPa':
        row['WVTR_Updated_Num'] = (row['WVP_Updated_Num'] * 101.325) / (row['thickness_Updated_Num'] * 1e6)
        row['WVTR_Updated_Unit'] = 'g/m2day'
        
    return row

df = df.apply(calculate_rates, axis=1)

# 5. Final Column Selection
# Keep only requested columns as per user requirements
final_columns = [
    'Base Material', 
    'Type', 
    'Secondary Material', 
    'WVTR_Updated_Num', 
    'OTR_Updated_Num', 
    'WVTR_Updated_Unit', 
    'OTR_Updated_Unit'
]

# Ensure columns exist before filtering
df_final = df[[col for col in final_columns if col in df.columns]]

# 6. Save Cleaned Dataset
df_final.to_csv('materials_permeability.csv', index=False)
print("Processing complete. Saved requested columns to 'cleaned_featured_data.csv'.")

Processing complete. Saved requested columns to 'cleaned_featured_data.csv'.


In [10]:
df = pd.read_csv('cleaned_featured_data.csv')
df

,Base Material,Type,Secondary Material,WVTR_Updated_Num,OTR_Updated_Num,WVTR_Updated_Unit,OTR_Updated_Unit
0,fish gelatin,individual,NaN,2.731398e+03,NaN,g/m2day,NaN
1,fish gelatin,nanocomposite,montmorillonite,7.091129e+02,NaN,g/m2day,NaN
2,fish gelatin,nanocomposite,montmorillonite,9.805018e+02,NaN,g/m2day,NaN
3,fish gelatin,nanocomposite,montmorillonite,1.199364e+03,NaN,g/m2day,NaN
4,fish gelatin,nanocomposite,montmorillonite,1.365699e+03,NaN,g/m2day,NaN
...,...,...,...,...,...,...,...
291,polyethylene terephthalate,individual,NaN,1.317224e+04,NaN,g/m2day,NaN
292,polypropylene,individual,NaN,4.995323e+08,NaN,g/m2day,NaN
293,polyvinylidene chloride,individual,NaN,7.082618e+08,NaN,g/m2day,NaN
294,EVOH,individual,NaN,9.625878e+07,NaN,g/m2day,NaN


In [14]:
# df[df['OTR_Updated_Num'].notna()]
df

,Base Material,Type,Secondary Material,WVTR_Updated_Num,OTR_Updated_Num,WVTR_Updated_Unit,OTR_Updated_Unit
0,fish gelatin,individual,NaN,2.731398e+03,NaN,g/m2day,NaN
1,fish gelatin,nanocomposite,montmorillonite,7.091129e+02,NaN,g/m2day,NaN
2,fish gelatin,nanocomposite,montmorillonite,9.805018e+02,NaN,g/m2day,NaN
3,fish gelatin,nanocomposite,montmorillonite,1.199364e+03,NaN,g/m2day,NaN
4,fish gelatin,nanocomposite,montmorillonite,1.365699e+03,NaN,g/m2day,NaN
...,...,...,...,...,...,...,...
291,polyethylene terephthalate,individual,NaN,1.317224e+04,NaN,g/m2day,NaN
292,polypropylene,individual,NaN,4.995323e+08,NaN,g/m2day,NaN
293,polyvinylidene chloride,individual,NaN,7.082618e+08,NaN,g/m2day,NaN
294,EVOH,individual,NaN,9.625878e+07,NaN,g/m2day,NaN
